#Установка зависимостей и импорт библиотек

In [17]:
!pip install playwright nest_asyncio pandas requests bs4
!playwright install --with-deps chromium

Installing dependencies...
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:3 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:4 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:6 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Fetched 3,917 B in 1s (3,198 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libasound2t64 is already the newest version (1.2.11-1ubuntu0.3).
libatk-bridge2.0-0t64 is already t

In [18]:
import asyncio, json, re, time, os
import requests
from bs4 import BeautifulSoup
import pandas as pd
from playwright.async_api import async_playwright, TimeoutError as PWTimeout
import nest_asyncio

# Применяем nest_asyncio для корректной работы в Jupyter
nest_asyncio.apply()

# Настройки отображения pandas
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 30)

#Диагностика источника

In [19]:
DYNAMIC_URL = "https://scrapingcourse.com/infinite-scrolling"

async def run_diagnostics():
    print("Ответ сервера")
    headers = {"User-Agent": "course-crawler/1.0"}

    t0 = time.perf_counter()
    resp = requests.get(DYNAMIC_URL, headers=headers, timeout=15)
    html_server = resp.text
    req_time = time.perf_counter() - t0

    soup = BeautifulSoup(html_server, "html.parser")
    products_server = soup.select(".product-item")

    print(f"Статус: {resp.status_code}")
    print(f"Размер HTML: {len(html_server)} байт")
    print(f"Время ответа: {req_time:.2f} сек")
    print(f"Найдено целевых элементов (товаров): {len(products_server)}")
    print("ВЫВОД: Статический парсер видит только первую порцию товаров\n")

    print("Поиск скрытого JSON в разметке")
    if "var data = [" in html_server or "application/ld+json" in html_server:
        print("В исходном HTML найдены блоки данных JSON")
    else:
        print("Массив со всеми товарами в исходном коде страницы не найден\n")

    print("Анализ Network/XHR")
    async with async_playwright() as pw:
        browser = await pw.chromium.launch(headless=True)
        context = await browser.new_context(user_agent=headers["User-Agent"])
        page = await context.new_page()

        seen_xhr = []

        # Расширенный перехватчик для сбора требуемой информации
        async def handle_response(response):
            req = response.request
            if req.resource_type in ("xhr", "fetch"):
                content_type = response.headers.get("content-type", "неизвестно")
                # Для GET параметры обычно в URL, для POST — в post_data
                params = req.post_data if req.post_data else "встроены в URL (GET)"

                seen_xhr.append({
                    "url": req.url,
                    "method": req.method,
                    "parameters": params,
                    "format": content_type,
                    "status": response.status
                })

        page.on("response", handle_response)

        await page.goto(DYNAMIC_URL, wait_until="domcontentloaded")

        print("Эмулируем прокрутку вниз для срабатывания JS-подгрузки")
        for _ in range(3):
            await page.mouse.wheel(0, 10000)
            await page.wait_for_timeout(1000)

        await browser.close()

        print(f"Зафиксировано фоновых запросов (XHR/fetch): {len(seen_xhr)}\n")
        print("Детали запросов с данными:")
        for idx, req_data in enumerate(seen_xhr[:5], 1):
            print(f"Запрос {idx}:")
            print(f"  URL:        {req_data['url']}")
            print(f"  Метод:      {req_data['method']}")
            print(f"  Параметры:  {req_data['parameters']}")
            print(f"  Формат:     {req_data['format']}")
            print(f"  Статус:     {req_data['status']}\n")

# Запуск функции диагностики
await run_diagnostics()

Ответ сервера
Статус: 200
Размер HTML: 18998 байт
Время ответа: 1.25 сек
Найдено целевых элементов (товаров): 12
ВЫВОД: Статический парсер видит только первую порцию товаров

Поиск скрытого JSON в разметке
Массив со всеми товарами в исходном коде страницы не найден

Анализ Network/XHR
Эмулируем прокрутку вниз для срабатывания JS-подгрузки
Зафиксировано фоновых запросов (XHR/fetch): 5

Детали запросов с данными:
Запрос 1:
  URL:        https://www.google-analytics.com/g/collect?v=2&tid=G-NZGD14H87G&gtm=45je6ad1v9183656648za208zd9183656648xf1&_p=1792071575042&gcd=13l3l3l3l1l1&npa=0&dma=0&cid=1365239567.1792071575&frm=0&ngs=1&pscdl=noapi&rcb=2&sr=1280x720&uaa=x86&uab=64&uafvl=Not%253AA-Brand%3B8.0.0.0%7CChromium%3B156.0.8078.4%7CHeadlessChrome%3B156.0.8078.4&uam=&uamb=0&uap=Windows&uapv=&uaw=0&ul=en-us&_s=1&tag_exp=115938465~115938468~118897920~118897930~120213116~120385422~120469145~120469153~121246026~121246034&sid=1792071575&sct=1&seg=0&dl=https%3A%2F%2Fwww.scrapingcourse.com%2Finfinit

In [20]:
STATIC_URL = "https://scrapingcourse.com/ecommerce"

def analyze_source(url, name):
    t0 = time.perf_counter()
    resp = requests.get(url, headers={"User-Agent": "course-crawler/1.0"}, timeout=15)
    req_time = time.perf_counter() - t0

    soup = BeautifulSoup(resp.text, "html.parser")
    # Ищем карточки товаров
    products = soup.select(".product")

    # Ищем элементы классической пагинации
    pagination = soup.select(".page-numbers, .pagination")
    has_pagination_links = any(a.has_attr('href') for p in pagination for a in p.find_all('a'))

    print(name)
    print(f"URL: {url}")
    print(f"Товаров в стартовом HTML: {len(products)}")
    print(f"Наличие HTML-пагинации: {has_pagination_links}")
    print(f"Время ответа сервера: {req_time:.2f} сек\n")

In [21]:
analyze_source(STATIC_URL, "Статический источник")

Статический источник
URL: https://scrapingcourse.com/ecommerce
Товаров в стартовом HTML: 16
Наличие HTML-пагинации: True
Время ответа сервера: 1.31 сек



#Каркас Playwright

In [22]:
async def test_initial_extraction():
    # Запуск Playwright и браузера
    pw = await async_playwright().start()
    browser = await pw.chromium.launch(headless=True)

    # Настройка контекста
    context = await browser.new_context(
        user_agent="course-crawler/1.0 (educational use)",
        viewport={"width": 1280, "height": 900},
        locale="ru-RU"
    )

    context.set_default_timeout(15_000)             # 15 секунд на любые действия
    context.set_default_navigation_timeout(15_000)  # 15 секунд на загрузку страницы

    page = await context.new_page()
    items = []

    try:
        print(f"Переход на страницу: {DYNAMIC_URL}")
        await page.goto(DYNAMIC_URL, wait_until="domcontentloaded")

        # 3. Ожидание первого элемента
        cards_locator = page.locator(".product-item")
        await cards_locator.first.wait_for(timeout=10_000)

        # 4. Извлечение первой страницы
        count = await cards_locator.count()

        for i in range(12):
            card = cards_locator.nth(i)

            # Семантические локаторы
            name_loc = card.locator(".product-name")
            name = await name_loc.inner_text() if await name_loc.count() > 0 else None

            link_loc = card.get_by_role("link").first
            link = await link_loc.get_attribute("href") if await link_loc.count() > 0 else None

            price_loc = card.locator(".product-price")
            price = await price_loc.inner_text() if await price_loc.count() > 0 else None

            items.append({
                "name": name,
                "price": price,
                "link": link
            })

    except PWTimeout:
        print("Ошибка таймаута: страница или элементы не загрузились вовремя")
    finally:
        # 5. Корректное закрытие ресурсов (обязательное требование)
        print("Закрытие ресурсов Playwright")
        await page.close()
        await context.close()
        await browser.close()
        await pw.stop()
        print("Ресурсы успешно освобождены")

    return items

In [23]:
initial_items = await test_initial_extraction()
df_initial = pd.DataFrame(initial_items)
display(df_initial)

Переход на страницу: https://scrapingcourse.com/infinite-scrolling
Закрытие ресурсов Playwright
Ресурсы успешно освобождены


,name,price,link
0,Chaz Kangeroo Hoodie,$52,https://scrapingcourse.com/ecommerce/product/c...
1,Teton Pullover Hoodie,$70,https://scrapingcourse.com/ecommerce/product/t...
2,Bruno Compete Hoodie,$63,https://scrapingcourse.com/ecommerce/product/b...
3,Frankie Sweatshirt,$60,https://scrapingcourse.com/ecommerce/product/f...
4,Hollister Backyard Sweatshirt,$52,https://scrapingcourse.com/ecommerce/product/h...
5,Stark Fundamental Hoodie,$42,https://scrapingcourse.com/ecommerce/product/s...
6,Hero Hoodie,$54,https://scrapingcourse.com/ecommerce/product/h...
7,Oslo Trek Hoodie,$42,https://scrapingcourse.com/ecommerce/product/o...
8,Abominable Hoodie,$69,https://scrapingcourse.com/ecommerce/product/a...
9,Mach Street Sweatshirt,$62,https://scrapingcourse.com/ecommerce/product/m...


#Браузерный сбор (Playwright)

In [24]:
# Блокировка тяжелых ресурсов
BLOCK = {"image", "media", "font"}
async def block_resources(route):
    if route.request.resource_type in BLOCK:
        return await route.abort()
    return await route.continue_()

async def locators_collection(max_items=200):
    os.makedirs("artifacts", exist_ok=True)

    pw = await async_playwright().start()
    browser = await pw.chromium.launch(headless=True)
    context = await browser.new_context(
        user_agent="course-crawler/1.0 (educational use)",
        viewport={"width": 1280, "height": 900},
        locale="ru-RU"
    )
    context.set_default_timeout(15_000)

    page = await context.new_page()
    await page.route("**/*", block_resources)

    item_selector = ".product-item"

    try:
        print(f"Запуск сбора (Метод: Локаторы). Цель: {max_items} записей...")
        await page.goto(DYNAMIC_URL, wait_until="domcontentloaded")
        await page.wait_for_selector(item_selector, timeout=10_000)
    except PWTimeout:
        await page.screenshot(path="artifacts/stage3_error.png")
        with open("artifacts/stage3_error.html", "w", encoding="utf-8") as f:
            f.write(await page.content())
        print("Ошибка первоначальной загрузки. Артефакты сохранены")
        await context.close(); await browser.close(); await pw.stop()
        return [], pd.DataFrame(), 0

    log = []
    max_rounds = 50

    t0_total = time.perf_counter()

    # Бесконечная прокрутка
    for r in range(max_rounds):
        try:
            # Уровень 3: ждем ЛЮБОЙ ответ от целевого API[cite: 1, 4]
            async with page.expect_response(lambda r: "/ajax/products" in r.url, timeout=4000) as caught:
                await page.mouse.wheel(0, 15_000)

            resp = await caught.value
            if resp.status == 200:
                pass # Данные пришли, продолжаем
            else:
                print(f"Пришел статус {resp.status}, выполнение завершено")
                break

        except PWTimeout:
            # Если запрос вообще не отправлялся фронтендом
            print("Таймаут ожидания: страница больше не запрашивает данные. Выполнение завершено")
            break

        cur_count = await page.locator(item_selector).count()
        log.append({"итерация": r + 1, "элементов_на_странице": cur_count})
        print(f"Итерация {r + 1}: элементов на странице = {cur_count}")

        if cur_count >= max_items:
            print(f"Сбор остановлен: собрано требуемое количество ({cur_count} >= {max_items}).")
            break

        await asyncio.sleep(0.5)

    print("\nНачинаем извлечение поэлементно (через локаторы)")
    t0_extract = time.perf_counter()

    items = []
    cards_locator = page.locator(item_selector)
    total_elements = min(await cards_locator.count(), max_items)

    for i in range(total_elements):
        card = cards_locator.nth(i)

        # Приоритет 5: Стабильные классы[cite: 1, 4]
        name_loc = card.locator(".product-name")
        name = await name_loc.inner_text() if await name_loc.count() > 0 else None

        price_loc = card.locator(".product-price")
        price = await price_loc.inner_text() if await price_loc.count() > 0 else None

        # Приоритет 2: Семантическая роль[cite: 1, 4]
        link_loc = card.get_by_role("link").first
        link = await link_loc.get_attribute("href") if await link_loc.count() > 0 else None

        items.append({
            "name": name,
            "price": price,
            "link": link
        })

    extract_time = time.perf_counter() - t0_extract
    total_time = time.perf_counter() - t0_total

    print(f"Время на извлечение: {extract_time:.2f} сек")

    await context.close()
    await browser.close()
    await pw.stop()

    return items[:max_items], pd.DataFrame(log), total_time, extract_time

In [25]:
# Запуск и сохранение результатов
items_locators, df_log_loc, time_total, time_ext = await locators_collection(max_items=200)

if items_locators:
    df_locators = pd.DataFrame(items_locators)
    # Чистим дубликаты
    df_locators = df_locators.drop_duplicates(subset=['link'], keep='first', ignore_index=True)

    print(f"\nИтог: Собрано {len(df_locators)} уникальных записей")
    print(f"Общее время скрипта: {time_total:.1f} сек")
    display(df_locators.head())

Запуск сбора (Метод: Локаторы). Цель: 200 записей...
Итерация 1: элементов на странице = 24
Итерация 2: элементов на странице = 36
Итерация 3: элементов на странице = 48
Итерация 4: элементов на странице = 60
Итерация 5: элементов на странице = 72
Итерация 6: элементов на странице = 84
Итерация 7: элементов на странице = 96
Итерация 8: элементов на странице = 108
Итерация 9: элементов на странице = 120
Итерация 10: элементов на странице = 132
Итерация 11: элементов на странице = 144
Итерация 12: элементов на странице = 156
Итерация 13: элементов на странице = 168
Итерация 14: элементов на странице = 180
Итерация 15: элементов на странице = 187
Пришел статус 204, выполнение завершено

Начинаем извлечение поэлементно (через локаторы)
Время на извлечение: 4.68 сек

Итог: Собрано 147 уникальных записей
Общее время скрипта: 16.5 сек


,name,price,link
0,Chaz Kangeroo Hoodie,$52,https://scrapingcourse.com/ecommerce/product/c...
1,Teton Pullover Hoodie,$70,https://scrapingcourse.com/ecommerce/product/t...
2,Bruno Compete Hoodie,$63,https://scrapingcourse.com/ecommerce/product/b...
3,Frankie Sweatshirt,$60,https://scrapingcourse.com/ecommerce/product/f...
4,Hollister Backyard Sweatshirt,$52,https://scrapingcourse.com/ecommerce/product/h...


In [26]:
BLOCK = {"image", "media", "font"}
async def block_resources(route):
    if route.request.resource_type in BLOCK:
        return await route.abort()
    return await route.continue_()

async def mass_collection(max_items=200):
    os.makedirs("artifacts", exist_ok=True)

    pw = await async_playwright().start()
    browser = await pw.chromium.launch(headless=True)
    context = await browser.new_context(
        user_agent="course-crawler/1.0 (educational use)",
        viewport={"width": 1280, "height": 900},
        locale="ru-RU"
    )
    context.set_default_timeout(15_000)

    page = await context.new_page()
    await page.route("**/*", block_resources)

    item_selector = ".product-item"

    try:
        print(f"Запуск сбора. Цель: {max_items} записей")
        await page.goto(DYNAMIC_URL, wait_until="domcontentloaded")
        # Ожидание 2 уровня для стартовой страницы
        await page.wait_for_selector(item_selector, timeout=10_000)
    except PWTimeout:
        # Диагностируемость сбоев: сохраняем артефакты
        await page.screenshot(path="artifacts/stage3_error.png")
        with open("artifacts/stage3_error.html", "w", encoding="utf-8") as f:
            f.write(await page.content())
        print("Ошибка первоначальной загрузки. Артефакты сохранены.")
        await context.close(); await browser.close(); await pw.stop()
        return [], pd.DataFrame(), 0

    log, prev_count, stalls = [], 0, 0

    # Предохранители цикла
    max_rounds = 50
    stall_limit = 3

    t0 = time.perf_counter()

    # Бесконечная прокрутка
    for r in range(max_rounds):
        try:
            # Уровень 3: ожидание конкретного сетевого ответа
            # Мы ждем ответа от XHR-эндпоинта, который выявили на этапе диагностики
            async with page.expect_response(lambda r: "/ajax/products" in r.url and r.status == 200, timeout=4000) as caught:
                await page.mouse.wheel(0, 15_000)

            resp = await caught.value
            if resp.status == 200:
                pass
            else:
                print(f"Пришел статус {resp.status}, выполнение завершено")
                break
        except PWTimeout:
          break

        cur_count = await page.locator(item_selector).count()
        log.append({"итерация": r + 1, "элементов_на_странице": cur_count})
        print(f"Итерация {r + 1}: элементов на странице = {cur_count}")

        if cur_count == prev_count:
            stalls += 1
            if stalls >= stall_limit:
                print("Сбор остановлен: элементы больше не подгружаются")
                break
        else:
            stalls = 0

        prev_count = cur_count

        if cur_count >= max_items:
            print(f"Сбор остановлен: собрано требуемое количество ({cur_count} >= {max_items})")
            break

        await asyncio.sleep(0.5)

    # --- МАССОВОЕ ИЗВЛЕЧЕНИЕ (eval_on_selector_all) ---
    t0_eval_extract = time.perf_counter()
    items = await page.eval_on_selector_all(item_selector, """
        els => els.map(el => ({
            name:  el.querySelector('.product-name')?.innerText ?? null,
            price: el.querySelector('.product-price')?.innerText?? null,
            link:  el.querySelector('a')?.href ?? null
        }))
    """)

    browser_time = time.perf_counter() - t0
    extract_time_browser = time.perf_counter() - t0_eval_extract
    print(f"Время на извлечение: {extract_time_browser:.2f} сек")

    await context.close()
    await browser.close()
    await pw.stop()

    return items[:max_items], pd.DataFrame(log), browser_time

In [27]:
items_browser, df_log, time_browser = await mass_collection(max_items=200)

if items_browser:
    df_browser = pd.DataFrame(items_browser)
    # Выгрузка результатов в файлы[cite: 2]
    df_browser.to_csv("data_browser.csv", index=False)
    df_log.to_csv("collection_log.csv", index=False)

    print(f"\nИтог: Собрано {len(items_browser)} записей за {time_browser:.1f} сек")
    print("Данные сохранены в 'data_browser.csv', журнал в 'collection_log.csv'")
    display(df_browser.head())
    display(df_log.tail())

Запуск сбора. Цель: 200 записей
Итерация 1: элементов на странице = 24
Итерация 2: элементов на странице = 36
Итерация 3: элементов на странице = 48
Итерация 4: элементов на странице = 60
Итерация 5: элементов на странице = 72
Итерация 6: элементов на странице = 84
Итерация 7: элементов на странице = 96
Итерация 8: элементов на странице = 108
Итерация 9: элементов на странице = 120
Итерация 10: элементов на странице = 132
Итерация 11: элементов на странице = 144
Итерация 12: элементов на странице = 156
Итерация 13: элементов на странице = 168
Итерация 14: элементов на странице = 180
Итерация 15: элементов на странице = 187
Время на извлечение: 0.04 сек

Итог: Собрано 187 записей за 14.8 сек
Данные сохранены в 'data_browser.csv', журнал в 'collection_log.csv'


,name,price,link
0,Chaz Kangeroo Hoodie,$52,https://scrapingcourse.com/ecommerce/product/c...
1,Teton Pullover Hoodie,$70,https://scrapingcourse.com/ecommerce/product/t...
2,Bruno Compete Hoodie,$63,https://scrapingcourse.com/ecommerce/product/b...
3,Frankie Sweatshirt,$60,https://scrapingcourse.com/ecommerce/product/f...
4,Hollister Backyard Sweatshirt,$52,https://scrapingcourse.com/ecommerce/product/h...


,итерация,элементов_на_странице
10,11,144
11,12,156
12,13,168
13,14,180
14,15,187


#Дешевая альтернатива (XHR / прямые HTTP запросы)

In [28]:
def light_collection(max_items=200, pause=0.5):
    #  Сбор данных через прямое обращение к XHR-эндпоинту пагинации
    s = requests.Session()
    s.headers.update({"User-Agent": "course-crawler/1.0 (educational use)"})

    items = []
    offset = 0  # Начальное смещение

    print("Запуск дешёвой альтернативы (requests + XHR API)")
    t0 = time.perf_counter()

    # Цикл пагинации через XHR-эндпоинт
    while len(items) < max_items:
        # Подставляем offset в эндпоинт, который мы нашли на Этапе 1
        api_url = f"https://www.scrapingcourse.com/ajax/products?offset={offset}"

        resp = s.get(api_url, timeout=10)
        if resp.status_code != 200:
            print(f"Остановка: сервер вернул код {resp.status_code}")
            break

        # Формат ответа — HTML (как мы выяснили на Этапе 1), поэтому парсим через BS4
        soup = BeautifulSoup(resp.text, "html.parser")
        products = soup.select(".product-item")

        if not products:
            print("Сервер вернул пустой список. Достигнут конец каталога")
            break

        for card in products:
            name_el = card.select_one(".product-name")
            price_el = card.select_one(".product-price")
            link_el = card.select_one("a")

            items.append({
                "name": name_el.get_text(strip=True) if name_el else None,
                "price": price_el.get_text(strip=True) if price_el else None,
                "link": link_el.get("href") if link_el else None
            })

        print(f"Смещение {offset:>3}: получено {len(products)} товаров. Всего в массиве: {len(items)}")

        offset += 10  # Увеличиваем смещение на размер порции
        if len(items) >= max_items:
            break

        time.sleep(pause)  # Пауза вежливости

    req_time = time.perf_counter() - t0
    return items[:max_items], req_time

In [29]:
# Запуск сбора
items_light, time_light = light_collection(max_items=200)

# Сохранение результата
df_light = pd.DataFrame(items_light)
df_light.to_csv("data_light.csv", index=False)

print(f"\nДешевой альтернативой собрано {len(df_light)} записей за {time_light:.2f} сек")

Запуск дешёвой альтернативы (requests + XHR API)
Смещение   0: получено 12 товаров. Всего в массиве: 12
Смещение  10: получено 12 товаров. Всего в массиве: 24
Смещение  20: получено 12 товаров. Всего в массиве: 36
Смещение  30: получено 12 товаров. Всего в массиве: 48
Смещение  40: получено 12 товаров. Всего в массиве: 60
Смещение  50: получено 12 товаров. Всего в массиве: 72
Смещение  60: получено 12 товаров. Всего в массиве: 84
Смещение  70: получено 12 товаров. Всего в массиве: 96
Смещение  80: получено 12 товаров. Всего в массиве: 108
Смещение  90: получено 12 товаров. Всего в массиве: 120
Смещение 100: получено 12 товаров. Всего в массиве: 132
Смещение 110: получено 12 товаров. Всего в массиве: 144
Смещение 120: получено 12 товаров. Всего в массиве: 156
Смещение 130: получено 12 товаров. Всего в массиве: 168
Смещение 140: получено 7 товаров. Всего в массиве: 175
Остановка: сервер вернул код 204

Дешевой альтернативой собрано 175 записей за 9.63 сек


#smoke-check

In [30]:
def smoke_check(df, dataset_name=""):
    print(f"Smoke-check: {dataset_name}")
    total = len(df)

    if total == 0:
        print("Ошибка: Датасет пуст!\n")
        return

    # Считаем пустые значения (NaN или пустые строки)
    empty_counts = {
        'name': df['name'].isna().sum() + (df['name'] == '').sum(),
        'price': df['price'].isna().sum() + (df['price'] == '').sum(),
        'link': df['link'].isna().sum() + (df['link'] == '').sum()
    }

    for field, count in empty_counts.items():
        empty_percent = (count / total) * 100
        if empty_percent > 5:
            # Выводим предупреждение при превышении порога в 5%
            print(f"Поле '{field}' превысило порог пустых значений "
                  f"({empty_percent:.1f}% — {count}/{total} записей)")
        else:
            print(f"Поле '{field}': пустых {empty_percent:.1f}% ({count}/{total}) - в пределах нормы")

    duplicates = df.duplicated().sum()
    if duplicates > 0:
        print(f"Найдено дубликатов: {duplicates}")
    else:
        print(f"Дубликатов нет.")

    print(f"Итого записей: {total}\n")

In [31]:
smoke_check(df_browser, "Браузерный набор (Playwright)")
smoke_check(df_light, "Дешёвая альтернатива (XHR + Requests)")

Smoke-check: Браузерный набор (Playwright)
Поле 'name': пустых 0.0% (0/187) - в пределах нормы
Поле 'price': пустых 0.0% (0/187) - в пределах нормы
Поле 'link': пустых 0.0% (0/187) - в пределах нормы
Найдено дубликатов: 40
Итого записей: 187

Smoke-check: Дешёвая альтернатива (XHR + Requests)
Поле 'name': пустых 0.0% (0/175) - в пределах нормы
Поле 'price': пустых 0.0% (0/175) - в пределах нормы
Поле 'link': пустых 0.0% (0/175) - в пределах нормы
Найдено дубликатов: 28
Итого записей: 175



In [32]:
browser_names = set(df_browser['name'].dropna())
light_names = set(df_light['name'].dropna())

print(f"Уникльных записей в браузере (Playwright) : {len(browser_names)}")
print(f"Записей через API (Requests)    : {len(light_names)}")

print("\nРезультат сверки:")
if len(browser_names) == len(light_names):
    print("Наборы абсолютно идентичны по количеству уникальных товаров")
else:
    print("Есть расхождения в составе данных:")

Уникльных записей в браузере (Playwright) : 147
Записей через API (Requests)    : 147

Результат сверки:
Наборы абсолютно идентичны по количеству уникальных товаров
